# Classifying direction

This notebook trains a classifier to predict whether the price goes up, goes
down, or stays flat over the next `h` steps. A training row holds the features
at step `t` and the class of the return over `(t, t + h]`, paired by
[`forecast_pairs`](../functions_supervised/forecast_pairs.md). The
[labels guide](../labels.md) shows the targets for other kinds of prediction.

The market is synthetic: a feature `sig` shifts the mean of the next return,
and the volatility changes slowly over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from screamer import (LogReturn, RollingStd, BacktestPriceTarget, backtest_report,
                      forecast_pairs)

# sig is an AR(1) signal with unit variance. The log return at step t is
# vol[t - 1] * (0.25 * sig[t - 1] + noise), with a slowly varying volatility vol.
rng = np.random.default_rng(11)
n = 10000
sig, log_vol = np.zeros(n), np.zeros(n)
for t in range(1, n):
    sig[t] = 0.9 * sig[t - 1] + np.sqrt(1 - 0.9 ** 2) * rng.standard_normal()
    log_vol[t] = 0.995 * log_vol[t - 1] + 0.05 * rng.standard_normal()
vol = 0.01 * np.exp(log_vol)
r = np.zeros(n)
r[1:] = vol[:-1] * (0.25 * sig[:-1] + rng.standard_normal(n - 1))
close = 100 * np.exp(np.cumsum(r))
step = np.arange(n)

## Features

`X` has three columns: the signal `sig`, the return over the last `h` steps,
which carries no edge in this market, and the volatility expected over the next
`h` steps, `sqrt(h)` times the standard deviation of the last 20 one-step
returns.

In [ ]:
h = 5
r1 = LogReturn(1)(close)
X = np.column_stack([
    sig,
    LogReturn(h)(close),                 # return over the last h steps
    RollingStd(20)(r1) * np.sqrt(h),     # volatility expected over the next h steps
])

## Labels

The target is `LogReturn(h)(close)`, the return over `(t - h, t]`. Paired with
`count=h`, row `t` holds the return over `(t, t + h]`. Dividing it by the
volatility column of the same row gives the move in units of the volatility
expected when the horizon starts. A move within half of that is flat (`0`);
beyond it, the class is up (`+1`) or down (`-1`).

Without `dropna`, row `t` is step `t`. `known` marks the rows whose features
and label are both known.

In [ ]:
Xp, ret = forecast_pairs(X, LogReturn(h)(close), count=h)
z = ret / Xp[:, 2]                                # move in units of expected volatility
label = np.where(z > 0.5, 1, np.where(z < -0.5, -1, 0))
known = ~np.isnan(Xp).any(axis=1) & ~np.isnan(z)

## Class balance

The dead zone of half an expected move puts about a third of the rows in each
class.

In [ ]:
classes, counts = np.unique(label[known], return_counts=True)
fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(["down", "flat", "up"], counts, color=["#c0392b", "#bbbbbb", "#28a745"])
ax.bar_label(bars)
ax.set(title="rows per class", ylabel="rows")
fig.tight_layout()

## Train and test split

The first 60% of the steps are for training and the rest for testing. The
training rows stop `h` steps before the test block, so no training label uses a
return from the test block. The model is a logistic regression on standardized
features, with one set of coefficients per class.

In [ ]:
cut = int(0.6 * n)
train = known & (step < cut - h)
test = known & (step >= cut)

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(Xp[train], label[train])
print("coefficients (rows: down, flat, up; columns: sig, last return, volatility)")
print(model[-1].coef_.round(3))

The `sig` coefficient is positive for up and negative for down. The other two
features have coefficients close to zero.

## Hit rate

On the test block, the hit rate is the share of the up and down predictions
for which the return over the next `h` steps has the predicted sign.

In [ ]:
pred = model.predict(Xp[test])
calls = pred != 0
hit = np.mean(np.sign(ret[test][calls]) == pred[calls])
print(f"{calls.sum()} up or down predictions in {test.sum()} test rows, hit rate {hit:.1%}")
print(f"share of test rows with a positive return: {np.mean(ret[test] > 0):.1%}")

## From probabilities to positions

`predict_proba` gives the probability of each class. `p_up` is the probability
of up given that the price moves, `P(up) / (P(up) + P(down))`. The position is
long above 0.55, short below 0.45, and flat in between. It is computed at every
step whose features are known, from the features at that step.

In [ ]:
proba = np.full((n, 3), np.nan)
has_x = ~np.isnan(X).any(axis=1)
proba[has_x] = model.predict_proba(X[has_x])      # columns: down, flat, up
p_up = proba[:, 2] / (proba[:, 2] + proba[:, 0])
pos = np.where(p_up > 0.55, 1.0, np.where(p_up < 0.45, -1.0, 0.0))
print(f"test block: long {np.mean(pos[cut:] > 0):.0%}, short {np.mean(pos[cut:] < 0):.0%}, "
      f"flat {np.mean(pos[cut:] == 0):.0%} of the steps")

## Backtest on the test block

`BacktestPriceTarget` holds one unit long or short with a fee of 5 basis points
on the traded notional. A position set at step `t` earns the price change from
`t` to `t + 1`. `backtest_report` summarizes the result.

In [ ]:
out = BacktestPriceTarget(fee=5e-4)(pos[cut:], close[cut:])
running, summary = backtest_report(out)
for key in ["total_pnl", "total_cost", "num_trades", "win_rate", "max_drawdown"]:
    print(f"{key:13s} {summary[key]:.4g}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True, height_ratios=[2, 1])
ax[0].plot(step[cut:], running["equity"], color="#0074a2")
ax[0].axhline(0, color="k", lw=0.6)
ax[0].set(title="equity on the test block", ylabel="cumulative pnl")
ax[1].plot(step[cut:], close[cut:], color="#555555", lw=0.8)
ax[1].set(xlabel="step", ylabel="price")
fig.tight_layout()